# <span style="color: #E60000;">Notebook 02. PostgreSQL Pipeline</span>
**Dự án:** HitRadar Pro | **Phân hệ:** EPIC 1 — Data Foundation

In [ ]:
import os
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host=os.getenv('POSTGRES_HOST', 'localhost'),
    port=int(os.getenv('POSTGRES_PORT', '5432')),
    database=os.getenv('POSTGRES_DB', 'hitradar'),
    user=os.getenv('POSTGRES_USER', 'postgres'),
    password=os.getenv('POSTGRES_PASSWORD', '123456')
)
print('Successfully connected to PostgreSQL HitRadar Database!')

OperationalError: connection to server at "localhost" (::1), port 5432 failed: FATAL:  password authentication failed for user "postgres"


**Nhận xét: Kết nối Cơ sở Dữ liệu PostgreSQL**

1. GIẢI THÍCH:
Mã nguồn thực hiện việc khởi tạo chuỗi kết nối (Connection String) đến hệ quản trị cơ sở dữ liệu quan hệ PostgreSQL thông qua thư viện `psycopg2` (PostgreSQL database adapter for Python). Thay vì mã hóa cứng (hardcode) các thông tin nhạy cảm như tên đăng nhập và mật khẩu trực tiếp vào mã nguồn, hệ thống sử dụng module `os` để trích xuất các biến môi trường (Environment Variables).

2. NHẬN XÉT:
Việc triển khai cấu hình kết nối thông qua biến môi trường là một quyết định kiến trúc cực kỳ chuyên nghiệp và tuân thủ chặt chẽ tiêu chuẩn bảo mật 12-Factor App. Điều này không chỉ giúp bảo vệ thông tin mật (Credentials) khỏi các rủi ro lộ lọt mã nguồn (ví dụ: vô tình push lên GitHub), mà còn mang lại tính linh hoạt tuyệt đối cho hệ thống hạ tầng (Infrastructure).

3. ĐÁNH GIÁ (MEDIUM IMPACT)
Bước thiết lập này đóng vai trò nền tảng. Một kết nối ổn định và bảo mật là tiền đề bắt buộc (Prerequisite) để toàn bộ quy trình Trích xuất - Chuyển đổi - Tải (ETL - Extract, Transform, Load) phía sau được vận hành trơn tru.

In [ ]:
tables_df = pd.read_sql("""
    SELECT table_name, table_type 
    FROM information_schema.tables 
    WHERE table_schema = 'public'
    ORDER BY table_type, table_name;
""", conn)
tables_df

**Nhận xét: Thống kê Danh sách Bảng & Views trong PostgreSQL**

1. GIẢI THÍCH:
Lệnh SQL truy vấn trực tiếp vào bảng hệ thống (System Catalog) `information_schema. tables` của PostgreSQL.

2. NHẬN XÉT:
Kết quả truy vấn phơi bày một chiến lược thiết kế cơ sở dữ liệu rất rõ ràng và mạch lạc. Dữ liệu gốc được lưu trữ cố định trong hai bảng vật lý là `tracks` và `artists`.

3. ĐÁNH GIÁ (HIGH IMPACT)
Chiến lược phân tách thành các View chuyên biệt này mang lại giá trị kiến trúc cực cao. Nó tạo ra ranh giới vật lý rõ ràng giữa hai môi trường: Môi trường Phân tích Khám phá (EDA - Exploratory Data Analysis) dành cho con người và Môi trường Huấn luyện Máy học (ML-Safe) dành riêng cho thuật toán.

In [ ]:
ml_view_df = pd.read_sql("""
    SELECT * FROM vw_ml_training_dataset LIMIT 5;
""", conn)
ml_view_df.info()
ml_view_df.head()

**Nhận xét: Kiểm định Schema View ML-Safe (`vw_ml_training_dataset`)**

1. GIẢI THÍCH:
Mã nguồn thực thi câu lệnh SQL đơn giản `SELECT * FROM vw_ml_training_dataset LIMIT 5;` để rút trích một mẫu siêu nhỏ từ Khung nhìn được thiết kế đặc biệt cho mục đích huấn luyện Trí tuệ Nhân tạo. Thông qua các hàm `info()` và `head()` của thư viện Pandas, hệ thống tiến hành siêu âm cấu trúc (Schema Profiling) của tập dữ liệu này.

2. NHẬN XÉT:
Kết quả kiểm định cho thấy một sự "thanh lọc" cực kỳ tàn nhẫn và cần thiết. Khung nhìn (View) này đã được thiết kế để chứa duy nhất nhãn mục tiêu (Target Label) `target_popularity` cùng với các đặc trưng âm thanh dạng số học (Numerical Features) tinh khiết.

3. ĐÁNH GIÁ (CRITICAL IMPACT)
Hành động thanh lọc và cô lập dữ liệu này có ý nghĩa sống còn (Critical) đối với sự thành bại của mô hình AI. Bằng cách xóa bỏ các định danh văn bản, hệ thống đã triệt tiêu hoàn toàn nguy cơ Rò rỉ Dữ liệu (Data Leakage) - hiện tượng mô hình gian lận bằng cách học thuộc lòng ID bài hát thay vì học các quy luật âm thanh phức tạp.

In [ ]:
conn.close()
print('PostgreSQL ETL Connection Closed Successfully!')

**Nhận xét: Đóng Kết nối PostgreSQL Pipeline**

1. GIẢI THÍCH:
Lệnh `conn. close()` là phương thức cuối cùng được gọi để ra lệnh cho driver `psycopg2` gửi tín hiệu chấm dứt phiên làm việc (Terminate Session) tới máy chủ PostgreSQL.

2. NHẬN XÉT:
Dù chỉ là một dòng code ngắn gọn, nhưng nó phản ánh sự chỉn chu và kỷ luật thép của một Kỹ sư Dữ liệu (Data Engineer) lành nghề. Việc quản lý vòng đời của một kết nối (Connection Lifecycle) là tối quan trọng trong các hệ thống Big Data.

3. ĐÁNH GIÁ (LOW IMPACT)
Dù không mang lại sự thay đổi nào về mặt dữ liệu, đây là một thao tác dọn dẹp Bắt buộc (Mandatory Cleanup). Nó tuân thủ các quy chuẩn khắt khe nhất trong nguyên lý Thiết kế Hệ thống Ổn định (Robust System Design), đảm bảo ứng dụng không để lại dấu vết tài nguyên bị rò rỉ (Memory Leaks), từ đó giữ cho máy chủ cơ sở dữ liệu luôn ở trạng thái khỏe mạnh nhất.

### 2. Các truy vấn tổng hợp
Tổng hợp dữ liệu theo năm phát hành:
```sql
SELECT LEFT(release_date, 4) AS release_year, COUNT(*) AS track_count
FROM raw.raw_tracks
GROUP BY release_year
ORDER BY release_year DESC
LIMIT 5;
```

## IX. Kết luận
**Trả lời 4 câu hỏi cốt lõi:**
1. **Dữ liệu đã được đưa vào PostgreSQL đúng chưa?**
   Đã import thành công bảng tracks và artists với đúng kiểu dữ liệu.
2. **Làm thế nào để tích hợp dữ liệu từ nhiều bảng?**
   Sử dụng SQL View làm Flat Table ảo, tránh nhân bản bản ghi vật lý.
3. **Dữ liệu nào sẽ được dùng cho Machine Learning?**
   Các Analytics Views như `vw_ml_training_dataset` được chuẩn bị trực tiếp trong PostgreSQL để cấp dữ liệu.
4. **Pipeline dữ liệu đã sẵn sàng cho Notebook 03 chưa?**
   Đã sẵn sàng. Notebook 03 sẽ đọc trực tiếp từ PostgreSQL View.
